# Day 18: Pandas GroupBy and Aggregation

Today we will summarize air-quality measurements by city and country. `groupby()` is useful when we need insights for each category instead of each individual row.

## 1. Load and inspect the dataset

The data contains pollution and weather measurements for several cities.

In [ ]:
import pandas as pd

air_quality = pd.read_csv('global_air_quality_data_10000.csv')
air_quality.head()

In [ ]:
print(f'Dataset shape: {air_quality.shape}')
print('\nColumns:')
print(air_quality.columns.tolist())
print('\nCities in the dataset:')
print(air_quality['City'].unique())

## 2. Group data by one column

`groupby('City')` creates one group for every city. We then apply an aggregation such as `mean()`, `max()`, or `count()`.

In [ ]:
# Average PM2.5 concentration for each city
average_pm25_by_city = (
    air_quality.groupby('City')['PM2.5']
    .mean()
    .sort_values(ascending=False)
)
average_pm25_by_city

In [ ]:
# Number of observations recorded for each country
observations_by_country = air_quality.groupby('Country').size().sort_values(ascending=False)
observations_by_country

## 3. Aggregate several columns

Pass a list of columns before calling an aggregation method to calculate the same statistic for several measurements.

In [ ]:
# Mean pollution and weather measurements by country
country_averages = (
    air_quality.groupby('Country')[['PM2.5', 'PM10', 'NO2', 'Temperature', 'Humidity']]
    .mean()
    .round(2)
)
country_averages

## 4. Use `agg()` for different calculations

`agg()` lets us apply different functions to different columns and give the result clear names.

In [ ]:
city_summary = (
    air_quality.groupby('City')
    .agg(
        records=('City', 'size'),
        average_pm25=('PM2.5', 'mean'),
        maximum_pm10=('PM10', 'max'),
        average_temperature=('Temperature', 'mean')
    )
    .round(2)
    .sort_values('average_pm25', ascending=False)
)
city_summary

## 5. Group by more than one column

Grouping by both country and city creates a hierarchical index. `reset_index()` turns those index levels back into ordinary columns.

In [ ]:
country_city_pm25 = (
    air_quality.groupby(['Country', 'City'], as_index=False)['PM2.5']
    .mean()
    .rename(columns={'PM2.5': 'average_pm25'})
    .sort_values('average_pm25', ascending=False)
    .round(2)
)
country_city_pm25.head(10)

## 6. Mini project: find the cities with the highest pollution

We will combine grouping, aggregation, sorting, and filtering to identify cities with high average PM2.5 levels.

In [ ]:
pollution_report = (
    air_quality.groupby(['Country', 'City'], as_index=False)
    .agg(
        average_pm25=('PM2.5', 'mean'),
        average_pm10=('PM10', 'mean'),
        average_no2=('NO2', 'mean')
    )
    .round(2)
    .sort_values('average_pm25', ascending=False)
)

top_5_polluted_cities = pollution_report.head(5)
top_5_polluted_cities

In [ ]:
overall_pm25 = air_quality['PM2.5'].mean()
above_average_cities = pollution_report.query('average_pm25 > @overall_pm25')

print(f'Overall average PM2.5: {overall_pm25:.2f}')
print(f'Cities above the overall average: {len(above_average_cities)}')
above_average_cities

## Practice

1. Find the minimum and maximum humidity for each city.
2. Calculate the average wind speed for every country.
3. Create a summary that includes the mean `O3` and the maximum `SO2` for each city.
4. Which city has the highest average `NO2` concentration?

## Key takeaways

- Use `groupby()` to split data into meaningful categories.
- Use `mean()`, `sum()`, `min()`, `max()`, and `size()` to summarize each group.
- Use `agg()` when each column needs a different aggregation.
- Sort grouped results to make comparisons easier.

## 7. Dataset operations

The following cells apply common Pandas operations to this dataset without changing the original `air_quality` DataFrame.

In [ ]:
# Data types, missing values, and duplicate rows
print(air_quality.dtypes)
print('\nMissing values per column:')
print(air_quality.isna().sum())
print(f'\nDuplicate rows: {air_quality.duplicated().sum()}')

In [ ]:
# Select columns and filter records
pollution_columns = air_quality[['City', 'Country', 'PM2.5', 'PM10', 'NO2', 'SO2', 'O3']]
high_pm25_records = air_quality.loc[air_quality['PM2.5'] > air_quality['PM2.5'].mean()]
print(f'Records with above-average PM2.5: {len(high_pm25_records)}')
high_pm25_records.head()

In [ ]:
# Sort records and calculate descriptive statistics
highest_pm25_records = air_quality.sort_values('PM2.5', ascending=False).head(10)
numeric_summary = air_quality.select_dtypes(include='number').describe().round(2)
highest_pm25_records[['City', 'Country', 'PM2.5']]

In [ ]:
# Practice 1: minimum and maximum humidity for each city
humidity_by_city = (
    air_quality.groupby('City', as_index=False)['Humidity']
    .agg(minimum_humidity='min', maximum_humidity='max')
    .sort_values('maximum_humidity', ascending=False)
)
humidity_by_city

In [ ]:
# Practice 2: average wind speed for every country
average_wind_speed_by_country = (
    air_quality.groupby('Country', as_index=False)['Wind Speed']
    .mean()
    .rename(columns={'Wind Speed': 'average_wind_speed'})
    .sort_values('average_wind_speed', ascending=False)
    .round(2)
)
average_wind_speed_by_country

In [ ]:
# Practice 3: mean O3 and maximum SO2 for each city
ozone_sulfur_summary = (
    air_quality.groupby('City', as_index=False)
    .agg(average_o3=('O3', 'mean'), maximum_so2=('SO2', 'max'))
    .sort_values('average_o3', ascending=False)
    .round(2)
)
ozone_sulfur_summary

In [ ]:
# Practice 4: city with the highest average NO2 concentration
highest_average_no2_city = (
    air_quality.groupby('City', as_index=False)['NO2']
    .mean()
    .rename(columns={'NO2': 'average_no2'})
    .sort_values('average_no2', ascending=False)
    .head(1)
    .round(2)
)
highest_average_no2_city